# Does a more flexible model do better?

Everything so far has used logistic regression. The Day 4 brief asks for two or three
different kinds of model to be tested against the same folds, using the same measures, before
committing to one. This notebook does that, and then makes the commitment.

The two challengers are both built from decision trees, which work in a completely different
way from logistic regression:

- **Logistic regression** adds up each column's effect and draws one straight dividing line.
  It cannot represent "these two things *together*" unless someone builds that column for it.
- **Random forest** grows hundreds of decision trees, each on a slightly different sample of
  the maps, and averages their votes.
- **Gradient boosting** (`HistGradientBoostingClassifier`) grows trees one after another, each
  trying to fix the mistakes of the ones before it.

Trees can learn combinations and bends that a straight line cannot. That makes this a second
test of something beyond "which model scores best": **if agent combinations matter, a tree
model should be able to find them where logistic regression could not.** Notebook 07 answered
the synergy question by analysing line-ups directly; this answers it a completely different way.

Each model is tried on three sets of columns:

| Set | Columns | What it is |
|---|---|---|
| without player stats | 39 | agents, map, who picked — notebook 04's model |
| with player stats | 2 | player-rating difference, who picked — notebook 06's model |
| everything | 41 | both together |

**What we expect.** With only two columns and 756 maps to learn from, a straight line is
probably the right shape — better-rated players, more likely to win — so logistic regression
should hold its own. If the trees win, that is interesting. If they lose, that is a finding
too: the simplest model is the right one.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from form import player_form
from model import evaluate, make_model, picker_baseline, summarise

maps = build_map_table()
learn, test = split_season(maps)

X_agents, y = build_features(learn, feature_columns(maps))
X = pd.concat([X_agents, player_form(learn, half_life=5)], axis=1)
agent_columns = [c for c in X_agents.columns if c.startswith("agent_")]
map_columns = [c for c in X_agents.columns if c.startswith("map_") and c != "map_picked_by_a"]

SETS = {
    "without player stats (39)": list(X_agents.columns),
    "with player stats (2)": ["rating_diff", "map_picked_by_a"],
    "everything (41)": list(X_agents.columns) + ["rating_diff"],
}

baseline = picker_baseline(y, learn)
print(f"learning half: {len(learn):,} maps   (test half and 2026 not touched)")
print(f"baseline to beat: {baseline['accuracy'].mean():.1%}")
print("log loss for saying 50/50 to every map: 0.693")

learning half: 756 maps   (test half and 2026 not touched)
baseline to beat: 55.5%
log loss for saying 50/50 to every map: 0.693


## A note on the machinery

Every model here goes through `src/model.py`, which wraps it in the same three steps inside
each fold: fill gaps with zero (a missing rating means "no difference between the teams"),
put the columns on a common scale, then fit. A scikit-learn `Pipeline` does the first two
using the training part of each fold only, so nothing about the maps being checked can leak
into the fitting.

The earlier notebooks hand-wrote that scaling each time. Before trusting the new code, it has
to reproduce their numbers exactly.

In [2]:
lr = make_model(LogisticRegression(max_iter=5000))
for name, expected in [("without player stats (39)", "51.1% / 0.760"),
                       ("with player stats (2)", "59.7% / 0.678")]:
    s = summarise(evaluate(lr, X[SETS[name]], y, learn), baseline)
    print(f"{name:28} {s['accuracy']:.1%} / {s['log loss']:.3f}    notebooks 04-06 got {expected}")

without player stats (39)    51.1% / 0.760    notebooks 04-06 got 51.1% / 0.760
with player stats (2)        59.7% / 0.678    notebooks 04-06 got 59.7% / 0.678


Identical. The new code changes nothing, which is what a refactor should do.

## 1. All three kinds of model, left on their default settings

Random forest and gradient boosting both involve some randomness, so each is given a fixed
seed (`random_state=0`) and will give the same answer every time it is run.

In [3]:
FAMILIES = {
    "logistic regression": lambda: LogisticRegression(max_iter=5000),
    "random forest": lambda: RandomForestClassifier(random_state=0),
    "gradient boosting": lambda: HistGradientBoostingClassifier(random_state=0),
}

rows = []
for set_name, cols in SETS.items():
    for family, build in FAMILIES.items():
        s = summarise(evaluate(make_model(build()), X[cols], y, learn), baseline)
        rows.append({"columns": set_name, "model": family, **s})

defaults = pd.DataFrame(rows)
shown = defaults.copy()
for c in ["accuracy", "worst fold", "best fold"]:
    shown[c] = (shown[c] * 100).round(1)
shown["log loss"] = shown["log loss"].round(3)
print(shown.to_string(index=False))

                  columns               model  accuracy  worst fold  best fold  log loss  beats baseline
without player stats (39) logistic regression      51.1        47.3       54.3     0.760               2
without player stats (39)       random forest      52.4        45.7       56.6     0.774               1
without player stats (39)   gradient boosting      52.9        44.0       59.5     0.773               2
    with player stats (2) logistic regression      59.7        56.5       64.1     0.678               4
    with player stats (2)       random forest      52.8        47.6       59.0     1.635               2
    with player stats (2)   gradient boosting      52.1        48.4       58.8     0.736               0
          everything (41) logistic regression      53.9        50.4       62.9     0.747               2
          everything (41)       random forest      53.6        45.7       60.3     0.725               2
          everything (41)   gradient boosting      53.8

**Only one of the nine combinations beats the baseline: logistic regression on two columns.**

Two things stand out.

**The trees fall apart on the two-column set** — both land *below* the baseline. With two
columns and 756 maps, the right shape really is a straight line. Trees instead chop the maps
into small boxes and learn whatever noise happens to sit in each.

**Random forest's log loss is 1.635** — more than double what you get by saying "50/50" to
every map. That number needs explaining rather than just reporting.

In [4]:
for family in ["logistic regression", "random forest"]:
    chances, truths = [], []
    for learn_rows, check_rows in season_folds(learn):
        cols = SETS["with player stats (2)"]
        fitted = clone(make_model(FAMILIES[family]())).fit(
            X.loc[learn_rows, cols].astype(float), y.loc[learn_rows])
        chances.append(fitted.predict_proba(X.loc[check_rows, cols].astype(float))[:, 1])
        truths.append(y.loc[check_rows].to_numpy())
    p, t = np.concatenate(chances), np.concatenate(truths)

    sure = (p <= 0.05) | (p >= 0.95)
    sure_and_wrong = sure & ((p >= 0.5).astype(int) != t)
    print(f"{family}:")
    print(f"   maps it was 95%+ sure about : {sure.mean():.0%}")
    print(f"   ...and got wrong            : {sure_and_wrong.sum()}")
    print(f"   its predictions ranged from : {p.min():.2f} to {p.max():.2f}\n")

logistic regression:
   maps it was 95%+ sure about : 0%
   ...and got wrong            : 0
   its predictions ranged from : 0.23 to 0.78



random forest:
   maps it was 95%+ sure about : 23%
   ...and got wrong            : 66
   its predictions ranged from : 0.00 to 1.00



**Random forest is confidently wrong.** It claims to be at least 95% sure about nearly a
quarter of the maps, and gets dozens of those wrong. Logistic regression never claims more
than about 78% about anything.

The cause is the default setting: each tree is grown until every leaf holds a single map, so
each tree effectively memorises its training maps and votes with total certainty. That is
exactly the failure log loss exists to catch. **Accuracy calls this model mediocre; log loss
shows it would actively mislead anyone who trusted it.** It is the clearest case in the
project for having chosen a second measure.

## 2. Giving the trees a fair shot

Defaults are an unfair test. Default random forest is set up in the way most prone to the
overconfidence just seen, so a fair comparison has to tune the trees.

Only the settings that control that overconfidence are tried:

- **Random forest** — `min_samples_leaf`, the fewest maps a leaf may hold (1 is the default),
  and `max_depth`, how many times a tree may split. 12 combinations.
- **Gradient boosting** — `max_depth`, `min_samples_leaf`, and `learning_rate` (smaller is
  more cautious). 12 combinations.

**How to read this honestly.** The trees get twelve attempts each; logistic regression gets
one, untuned. Trying many settings and keeping the best flatters whatever is being tuned — so
here that bias works *in the trees' favour*, deliberately. If the best of twelve still loses
to one untuned logistic regression, the result is not in doubt.

In [5]:
from itertools import product

GRIDS = {
    "random forest": [
        dict(min_samples_leaf=leaf, max_depth=depth)
        for leaf, depth in product([1, 10, 30, 60], [None, 3, 6])
    ],
    "gradient boosting": [
        dict(max_depth=depth, min_samples_leaf=leaf, learning_rate=rate)
        for depth, leaf, rate in product([None, 2, 3], [20, 50], [0.1, 0.03])
    ],
}
BUILD = {"random forest": RandomForestClassifier,
         "gradient boosting": HistGradientBoostingClassifier}

tuned = []
for set_name, cols in SETS.items():
    for family, grid in GRIDS.items():
        for params in grid:
            s = summarise(evaluate(make_model(BUILD[family](random_state=0, **params)),
                                   X[cols], y, learn), baseline)
            tuned.append({"columns": set_name, "model": family, "setting": params, **s})
tuned = pd.DataFrame(tuned)

lr_by_set = defaults[defaults["model"] == "logistic regression"].set_index("columns")

for set_name in SETS:
    lr_acc = lr_by_set.loc[set_name, "accuracy"]
    print(f"=== {set_name}    logistic regression, untuned: {lr_acc:.1%}")
    for family in GRIDS:
        f = tuned[(tuned["columns"] == set_name) & (tuned["model"] == family)]
        best = f.loc[f["accuracy"].idxmax()]
        print(f"   {family:18} {len(f)} settings, {f['accuracy'].min():.1%} to {f['accuracy'].max():.1%}"
              f"   | beat logistic regression: {(f['accuracy'] > lr_acc).sum():2}"
              f"   | log loss {f['log loss'].min():.3f} to {f['log loss'].max():.3f}")
        print(f"   {'':18} best {best['accuracy']:.1%} with {best['setting']}")
    print()

=== without player stats (39)    logistic regression, untuned: 51.1%
   random forest      12 settings, 51.6% to 53.6%   | beat logistic regression: 12   | log loss 0.691 to 0.774
                      best 53.6% with {'min_samples_leaf': 10, 'max_depth': 6}
   gradient boosting  12 settings, 52.5% to 56.5%   | beat logistic regression: 12   | log loss 0.697 to 0.773
                      best 56.5% with {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.1}

=== with player stats (2)    logistic regression, untuned: 59.7%
   random forest      12 settings, 52.0% to 58.2%   | beat logistic regression:  0   | log loss 0.673 to 1.635
                      best 58.2% with {'min_samples_leaf': 30, 'max_depth': None}
   gradient boosting  12 settings, 52.1% to 57.6%   | beat logistic regression:  0   | log loss 0.674 to 0.736
                      best 57.6% with {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.03}

=== everything (41)    logistic regression, untuned: 53.

**On the model that matters — two columns — not one of the twenty-four tree settings beats
logistic regression.** The best tree comes within a point and a half and no closer, despite
getting twelve attempts to logistic regression's one.

**Tuning did fix the overconfidence.** Requiring each leaf to hold at least 30 maps brings
random forest's log loss from 1.635 down to about 0.673 — as honest as logistic regression's
probabilities, and exactly the fix the diagnosis pointed to. It just is not as accurate.

**On the agent-heavy sets, almost every tree setting beats logistic regression.** That looks
like the trees finding something in the agents. The next section checks whether it is.

## 3. Is the tree finding the agents, or finding the picker?

The "without player stats" set contains the who-picked column as well as the agents. The
baseline *is* the who-picked rule. So a tree that scores just above the baseline on that set
might simply have rediscovered the rule, and be ignoring the agents entirely.

The direct test is to take pieces away and refit.

In [6]:
best_setting = (tuned[(tuned["columns"] == "without player stats (39)")
                      & (tuned["model"] == "gradient boosting")]
                .sort_values("accuracy").iloc[-1]["setting"])
print(f"best gradient-boosting setting on the agent set: {best_setting}\n")

pieces = {
    "agents + maps + who picked (all 39)": list(X_agents.columns),
    "maps + who picked, no agents": map_columns + ["map_picked_by_a"],
    "who picked only": ["map_picked_by_a"],
    "agents only, no who-picked": agent_columns,
}
for name, cols in pieces.items():
    s = summarise(evaluate(make_model(HistGradientBoostingClassifier(random_state=0, **best_setting)),
                           X[cols], y, learn), baseline)
    print(f"   {name:38} {s['accuracy']:.1%}   beats baseline on {s['beats baseline']}/5 folds")

best gradient-boosting setting on the agent set: {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.1}



   agents + maps + who picked (all 39)    56.5%   beats baseline on 3/5 folds


   maps + who picked, no agents           55.3%   beats baseline on 1/5 folds


   who picked only                        55.5%   beats baseline on 0/5 folds


   agents only, no who-picked             49.0%   beats baseline on 0/5 folds


**The tree's score is almost all the who-picked rule.** Take the agents away and it barely
moves. And given the agents *alone* — with a model perfectly capable of learning combinations
of them — it scores **below a coin flip.**

Adding the agents to the picker is worth about a point. That point sits well inside the
margin of error, and it comes from the best of twelve settings, so it cannot be told apart
from nothing.

One more view, because it can look like it disagrees. Instead of refitting without a column,
you can take the fitted model and scramble one column at a time on maps it has not seen, to
see how much it *leans on* each.

In [7]:
cols = list(X_agents.columns)
leaning = pd.Series(0.0, index=cols)
for learn_rows, check_rows in season_folds(learn):
    fitted = clone(make_model(HistGradientBoostingClassifier(random_state=0, **best_setting))).fit(
        X.loc[learn_rows, cols].astype(float), y.loc[learn_rows])
    result = permutation_importance(fitted, X.loc[check_rows, cols].astype(float),
                                    y.loc[check_rows], n_repeats=10, random_state=0)
    leaning += pd.Series(result.importances_mean, index=cols) / 5

print("accuracy lost when one column is scrambled, averaged over the folds:\n")
for column, lost in leaning.sort_values(ascending=False).head(5).items():
    print(f"   {column:20} {lost * 100:+.2f} points")
print(f"\n   all 27 agent columns added together: {leaning[agent_columns].sum() * 100:+.2f} points")

accuracy lost when one column is scrambled, averaged over the folds:

   map_picked_by_a      +4.52 points
   agent_tejo           +1.26 points
   agent_vyse           +1.13 points
   agent_breach         +1.03 points
   agent_neon           +0.84 points

   all 27 agent columns added together: +6.69 points


Who-picked is the single column the model leans on hardest. Each agent on its own is worth
about a point — roughly one map in a fold of 120.

Added together, the agents look larger than the picker. That is not a contradiction with the
refitting test above, and the difference between the two is worth understanding. **Scrambling**
a column measures how much *this particular fitted model* depends on it: if the model has
built its splits around the agents, scrambling them sends maps down the wrong branches, and it
suffers. **Refitting without** a column measures how much that column *adds*: the model gets
to rebuild itself around what is left. A model can depend heavily on something that is not
actually helping it. Refitting is the fair test of whether a column earns its place — and
refitting says the agents add about a point, which is noise.

**This is the same answer notebook 07 reached by a completely different route.** There, the
synergy analysis showed that a realistic agent effect is too small for one season of data to
detect. Here, the most flexible models available, tuned in their favour, extract about a point
from the agents. Two independent methods; one conclusion.

## 4. The decision

**Logistic regression on two columns — the players' rating difference and who picked the
map — is the final model.**

It is not a close call, and it is not chosen by default:

- **It is the most accurate.** Nothing else beats it on the columns that matter — not one
  of twenty-four tree settings, each given the chance to be tuned in its favour.
- **Its probabilities are among the most honest.** Log loss of 0.678, against 0.693 for knowing
  nothing. Tuned trees match it, at about 0.673 — but they are less accurate, and only after
  twelve attempts each.
- **It is the simplest and the easiest to explain.** Two numbers go in; each has one coefficient
  that says how much it matters. A non-technical audience can follow that.
- **It is the hardest to fool yourself with.** It has almost nothing to tune, so it has almost
  no room to have been fitted to the validation folds by accident.

## What this notebook found

1. **Only one combination of nine beats the baseline on defaults**: logistic regression on the
   two-column set.
2. **The tree models lose on the model that matters even when tuned in their favour.** Twenty-four
   settings, none beating one untuned logistic regression.
3. **Default random forest is confidently wrong** — 95%+ sure on nearly a quarter of maps and
   wrong on dozens of them. Log loss exposed it; accuracy could not have.
4. **Trees are better than logistic regression at *ignoring* useless columns**, which is why
   they win on the agent-heavy sets. That is not the same as finding value in them.
5. **Agent combinations add nothing this data can detect**, now shown by two independent
   methods.

The test half and the 2026 season are still untouched.